# RAG Basics

## Setup

In [1]:
# from IPython.display import Markdown, display
from pprint import pprint
import openai
import httpx
import os
import numpy as np 
import psycopg2
from pgvector.psycopg2 import register_vector

API_KEY = os.getenv("MODEL_API_KEY_DEV")
if not API_KEY:
    raise RuntimeError("Missing MODEL_API_KEY_DEV in environment. Restart the dev container after key generation/rotation.")

BASE_URL_CLEAN = os.getenv("MODEL_BASE_URL_CLEAN")
if not BASE_URL_CLEAN:
    raise RuntimeError("Missing MODEL_BASE_URL_CLEAN in environment. Restart the dev container after env updates.")

# local CA from your compose cert setup
CA_CERT_PATH = "/certs/.caroot/rootCA.pem"
VERIFY_CONFIG = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True

GENERATION_MODELS = ["ollama_chat/llama3.2:3b", "ollama_chat/llama3.2:1b"]
EMBEDDING_MODELS = ["ollama/nomic-embed-text:latest",
"ollama/qllama/bge-small-en-v1.5:q4_k_m"]

## Generate RAG content and embed

In [ ]:
def create_rag_content(model, question):
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    client = openai.OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL_CLEAN,
        http_client=http_client,
        max_retries=0,
    )
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": question}],
            temperature=0.0,
            stream=True,
            
        )
        print(f"Asking {model}: ", end="")

        parts = []
        for chunk in stream:
            if not getattr(chunk, "choices", None):
                continue
            delta = chunk.choices[0].delta.content
            if isinstance(delta, str):
                print(".", flush=True, end="")
                parts.append(delta)

        print(" Done.")
        return "".join(parts)
    except openai.OpenAIError as e:
        print(f"OpenAI API error creating RAG content: {e}")
        return None
    except Exception as e:
        print(f"Error creating RAG content: {e}")
        return None

def collect_rag_content(questions, generation_models):
    rag_content = []
    for question in questions:
        for model in generation_models:
            content = create_rag_content(model, question)
            if content:
                rag_content.append(content)
    return rag_content

questions = ["Gib genau 3 Vortragstitel zu KI aus. Ausgabeformat: Titel1$Titel2$Titel3. Keine weiteren Zeichen.", 
"Gib genau 3 Vortragstitel zu Rasenpflege aus. Ausgabeformat: Titel1$Titel2$Titel3. Keine weiteren Zeichen.",
"Gib genau 3 Vortragstitel zu Rasenpflege mit KI aus. Ausgabeformat: Titel1$Titel2$Titel3. Keine weiteren Zeichen.",] 

rag_raw_content = collect_rag_content(questions, GENERATION_MODELS)
rag_content = []
for content in rag_raw_content:
    print(f"content: {content}")
    if "$" in content:
        lines = content.split("$")
    elif "\n" in content:   
        lines = content.split("\n")
    elif "\t" in content:
        lines = content.split("\t")
    else:
        lines = [content]   

    for line in lines:
        line = line.replace("\n", "")
        line = line.replace("'", "")
        line = line.replace('"', '')
        line = line.strip()
        if line:
            print(f"line: {line}")
            rag_content.append(line)

pprint(rag_content)  
# display(Markdown(rag_content))


Asking ollama_chat/llama3.2:3b: ....................................................... Done.
Asking ollama_chat/llama3.2:1b: ............................................ Done.
Asking ollama_chat/llama3.2:3b: ............................................................. Done.
Asking ollama_chat/llama3.2:1b: .................................................... Done.
Asking ollama_chat/llama3.2:3b: ........................................................ Done.
Asking ollama_chat/llama3.2:1b: ................................................................ Done.
content: "Die Zukunft der Arbeit: Wie KI unsere Welt verändert"

"KI und Ethik: Die Herausforderungen einer moralischen Intelligenz"

"Machine Learning: Von Daten zu Entscheidungen - Die Kunst der KI-Modellierung"
line: Die Zukunft der Arbeit: Wie KI unsere Welt verändert
line: KI und Ethik: Die Herausforderungen einer moralischen Intelligenz
line: Machine Learning: Von Daten zu Entscheidungen - Die Kunst der KI-Modellierung
conte

Embed RAG context

In [8]:
def embed_texts(texts, model):
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    client = openai.OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL_CLEAN,
        http_client=http_client,
    )

    try:
        response = client.embeddings.create(
            model=model,
            input=texts,
        )
        return [item.embedding for item in response.data]

    except openai.OpenAIError as e:
        print(f"OpenAI API error embedding texts: {e}")
        return None
    except Exception as e:
        print(f"Error embedding texts: {e}")
        return None
      

embeddings = embed_texts(rag_content, EMBEDDING_MODELS[0])
if embeddings:
    print(f"Rag content: {rag_content[0]}")
    print(f"Embedding: {embeddings[0]}")

try:
    conn = psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_TOOLBERT_DB"),
        user=os.getenv("POSTGRES_TOOLBERT_USER"),
        password=os.getenv("POSTGRES_TOOLBERT_PASSWORD"),
    )
    conn.autocommit = True
    register_vector(conn)

    with conn, conn.cursor() as cur:
        # Store embeddings in documents table
        for content, embedding in zip(rag_content, embeddings):
            cur.execute("""
                INSERT INTO toolbert.documents (source, content, embedding)
                VALUES (%s, %s, %s)
            """, ("notebook", content, embedding))

except psycopg2.OperationalError as e:
    print(f"Error connecting to PostgreSQL: {e}")
    exit(1)
except Exception as e:
    print(f"Error: {e}")
    exit(1)


Rag content: Die Zukunft der Arbeit: Wie KI unsere Welt verändert
Embedding: [-6.423315e-05, -0.019796459, -0.15534793, 0.019410893, 0.0071045593, 0.03420048, -0.033580832, 0.0050882692, -0.070354514, -0.04696209, -0.015630387, 0.05734551, 0.06211041, 0.022696856, 0.037639175, -0.049122322, 0.044519797, -0.07222354, -0.049858727, 0.051027935, 0.0009657071, 0.028611377, -0.009544559, -0.040643748, 0.106407635, 0.015970955, 0.051302522, 0.058399994, -0.054608647, 0.020899536, 0.012744188, -0.03176361, -0.032319393, 0.0009354148, 0.00026435682, -0.05743445, 0.02257185, 0.05089526, 0.043805636, -0.030593097, 0.02322217, -0.0030824097, -0.024802854, -0.023024483, 0.079091534, -0.011324928, 0.0027894657, 0.019517386, 0.08203044, -0.03842533, -0.022037275, 0.018723603, 0.03191785, -0.039753117, 0.05395038, -0.004544654, -0.040359158, -0.04373494, 0.039603405, 0.04576622, 0.04233826, 0.045935653, -0.016493093, 0.031127462, -0.013107594, -0.03270478, -0.03245573, 0.025677396, 0.068916775, -0.03

## Top 3 search results

In [ ]:
query_vec = np.array(
    embed_texts(["I am interested in AI talks"], EMBEDDING_MODELS[0])[0], 
    dtype=np.float32 )


with conn, conn.cursor() as cur:
    cur.execute("""
        SELECT source, content, embedding <-> %s::vector as distance
        FROM toolbert.documents
        ORDER BY distance
        LIMIT 10
    """, (query_vec.tolist(),))
    rows = cur.fetchall()

for i, (source, content, distance) in enumerate(rows, start=1):
    print(f"{i}. (distance={distance:.6f}) {source}: {content[:60]}")

1. (distance=0.951673) notebook: 1. Die Zukunft der Maschinenlehrer
2. (distance=0.957236) notebook: 2. Künstliche Intelligenz und soziale Verantwortung
3. (distance=0.977389) notebook: Gartenreinigung - Die Notwendigkeit einer regelmäßigen Pfleg


## RAG request

In [ ]:
retrieved_festival_talks = sorted_festival_talks[:3]


def chat_w_rag(model, question, rag_content):
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    client = openai.OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL_CLEAN,
        http_client=http_client,
        max_retries=0,
    )
    content = f"Nutzerfrage: {question}\n\nHier sind die am besten passenden Vorträge:\n\n " + "\n".join(rag_content) +  "\n\nAntworte auf die Nutzerfrage"
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": content}],
            temperature=0.0,
            stream=True,
            
        )
        print(f"\n\nAsking {model}:")


        for chunk in stream:
            if not getattr(chunk, "choices", None):
                continue
            delta = chunk.choices[0].delta.content
            if isinstance(delta, str):
                print(delta, flush=True, end="")
             

        print(" Done.")

    except openai.OpenAIError as e:
        print(f"OpenAI API error creating RAG content: {e}")
 
    except Exception as e:
        print(f"Error creating RAG content: {e}")

for model in GENERATION_MODELS:
    chat_w_rag(model, "Ich interessiere mich für Vorträge zu KI", retrieved_festival_talks)  